# Gemma 3 12B — QLoRA Fine-tune (Türkçe SaaS Asistanı)

**Hedef:** turer73 ekosistemi için özel asistan (Linux-AI Server, PetVet, Kuafor SaaS, Panola ERP).

**Gereksinimler:**
- Colab Pro veya Pro+ (A100 40GB önerilir, L4 24GB minimum)
- HuggingFace token (Gemma 3 erişimi için: https://huggingface.co/google/gemma-3-12b-it)
- Eğitim datası: `merged.jsonl` (messages formatında)

**Çıktı:** LoRA adapter + (opsiyonel) GGUF dosyası → Ollama'da SER8'de çalıştırılabilir.

---

## 1. Ortam kontrolü

In [ ]:
!nvidia-smi

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok — Runtime > Change runtime type > A100/L4 seç"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}")
print(f"VRAM: {gpu.total_memory / 1e9:.1f} GB")
if gpu.total_memory < 22e9:
    print("[UYARI] 24GB altı VRAM — batch_size=1, gradient_accumulation arttır")

## 2. Bağımlılıklar (Unsloth)

In [ ]:
%%capture
!pip install --upgrade pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets sentencepiece protobuf

## 3. HuggingFace login (Gemma 3 gated model)

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Colab Secrets > HF_TOKEN ekle (https://huggingface.co/settings/tokens — read scope yeter)
login(userdata.get('HF_TOKEN'))

## 4. Konfigürasyon

In [ ]:
CONFIG = {
    'model_name': 'unsloth/gemma-3-12b-it-bnb-4bit',
    'max_seq_length': 4096,
    'load_in_4bit': True,
    'lora_r': 32,
    'lora_alpha': 32,
    'lora_dropout': 0.0,
    'target_modules': [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    'batch_size': 2,
    'grad_accum': 4,
    'epochs': 2,
    'learning_rate': 2e-4,
    'warmup_steps': 10,
    'weight_decay': 0.01,
    'lr_scheduler': 'cosine',
    'seed': 3407,
    'output_dir': '/content/gemma3-12b-tr-lora',
    'dataset_path': '/content/merged.jsonl',
    'hf_repo': 'turer73/gemma3-12b-tr-saas',
}
print(CONFIG)

## 5. Model + tokenizer (4-bit)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,
    load_in_4bit=CONFIG['load_in_4bit'],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    target_modules=CONFIG['target_modules'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=CONFIG['seed'],
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

## 6. Dataset

**Beklenen format:** her satır bir JSON, `messages` alanı OpenAI chat formatında:

```json
{"messages": [
  {"role": "system", "content": "Sen turer73 ekosisteminin teknik asistanısın."},
  {"role": "user", "content": "PetVet'te yeni bir pet türü nasıl eklerim?"},
  {"role": "assistant", "content": "Cloudflare Workers'ta..."}
]}
```

Aşağıdaki hücre `merged.jsonl` dosyasını upload etmeni isteyecek. Veri henüz yoksa örnek datayla başla.

In [ ]:
import os
from google.colab import files

if not os.path.exists(CONFIG['dataset_path']):
    print('merged.jsonl upload et:')
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    os.rename(fname, CONFIG['dataset_path'])

with open(CONFIG['dataset_path']) as f:
    n_lines = sum(1 for _ in f)
print(f'Toplam örnek: {n_lines}')

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template='gemma-3')

raw = load_dataset('json', data_files=CONFIG['dataset_path'], split='train')
raw = standardize_sharegpt(raw)

def formatting(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

ds = raw.map(formatting, remove_columns=raw.column_names)
split = ds.train_test_split(test_size=0.05, seed=CONFIG['seed'])
train_ds, eval_ds = split['train'], split['test']

print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')
print('--- ÖRNEK ---')
print(train_ds[0]['text'][:800])

## 7. Trainer

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=CONFIG['max_seq_length'],
        per_device_train_batch_size=CONFIG['batch_size'],
        gradient_accumulation_steps=CONFIG['grad_accum'],
        warmup_steps=CONFIG['warmup_steps'],
        num_train_epochs=CONFIG['epochs'],
        learning_rate=CONFIG['learning_rate'],
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        eval_strategy='steps',
        eval_steps=50,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=2,
        optim='adamw_8bit',
        weight_decay=CONFIG['weight_decay'],
        lr_scheduler_type=CONFIG['lr_scheduler'],
        seed=CONFIG['seed'],
        output_dir=CONFIG['output_dir'],
        report_to='none',
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part='<start_of_turn>user\n',
    response_part='<start_of_turn>model\n',
)

## 8. Train

In [ ]:
stats = trainer.train()
print(stats)

In [ ]:
used = torch.cuda.max_memory_reserved() / 1e9
print(f'Peak VRAM: {used:.2f} GB')
print(f'Süre: {stats.metrics["train_runtime"] / 60:.1f} dk')

## 9. Hızlı eval (örnek inference)

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    'Linux-AI Server kernel modulleri ne ise yarar?',
    'PetVet veritabani semasi nasil tasarlanmis?',
    'Kuafor SaaS\'ta randevu cakismasi kontrolu nasil yapiliyor?',
    'Panola ERP\'de stok hareketi nasil kaydedilir?',
]

for q in test_prompts:
    msgs = [{'role': 'user', 'content': q}]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt',
    ).to('cuda')
    out = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'Q: {q}')
    print(f'A: {response}\n{"-"*60}')

## 10. Save (LoRA adapter)

In [ ]:
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print(f'LoRA adapter: {CONFIG["output_dir"]}')
!ls -lh {CONFIG['output_dir']}

## 11. (Opsiyonel) HuggingFace'e push

In [ ]:
# model.push_to_hub(CONFIG['hf_repo'], token=userdata.get('HF_TOKEN'), private=True)
# tokenizer.push_to_hub(CONFIG['hf_repo'], token=userdata.get('HF_TOKEN'), private=True)

## 12. (Opsiyonel) GGUF export — Ollama için

Bu adım merge + quantize yapar (q4_k_m önerilir, 12B → ~7GB). `.gguf` dosyasını SER8'e indirip `ollama create` ile yükle.

In [ ]:
# model.save_pretrained_gguf(
#     CONFIG['output_dir'] + '-gguf',
#     tokenizer,
#     quantization_method='q4_k_m',
# )
# !ls -lh {CONFIG['output_dir']}-gguf

## 13. SER8'e indirme talimatı

```bash
# Colab tarafında:
from google.colab import files
files.download('/content/gemma3-12b-tr-lora-gguf/unsloth.Q4_K_M.gguf')

# SER8 (klipper) tarafında:
scp gemma3-12b-tr.gguf klipperos@klipper:/opt/models/
ollama create gemma3-tr -f Modelfile
# Modelfile içeriği:
#   FROM /opt/models/gemma3-12b-tr.gguf
#   PARAMETER temperature 0.7
#   PARAMETER num_ctx 4096
ollama run gemma3-tr
```